# SOLPS plasma background → Hermes-3 grid file

**Goal:** take the converged SOLPS plasma (`SOLPS_Luciani_off/balance.nc`) as a *frozen* plasma
background for Hermes-3, so that Hermes-3 evolves **only the neutrals** on top of it.

In [5]:
%matplotlib inline
import os
import sys
import shutil

import numpy as np
import matplotlib.pyplot as plt
from boututils.datafile import DataFile

SDTOOLS = r"/Users/zero/workspace/phd_work/hms_output/sdtools"
sys.path.append(SDTOOLS)

from code_comparison.solps_pp import SOLPScase
from code_comparison.interpolate import interpolateSOLPStoHermes
from hermes3.load import Load
from hermes3.selectors import get_1d_radial_data, get_1d_poloidal_data
from hermes3.grid_fields import Mesh, Field
from hermes3.utils import constants
from hermes3.accessors import *   # registers the .hermes accessor used by the plots


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
ROOT = r"/Users/zero/workspace/phd_work/hms_output"

SOLPS_DIR = os.path.join(ROOT, "SOLPS_Luciani_off")                              # holds balance.nc
CASE_DIR = os.path.join(ROOT, "260717_highrsl_6e20_coreOO")                       # any run on the target grid
GRID_IN = os.path.join(ROOT, "gridfile", "CDN_46895_260513_nowallpump_6e20.nc")
GRID_OUT = os.path.join(ROOT, "gridfile", "CDN_46895_highrsl_6e20_solpsbg_260910_my_nonml.nc")    


# ---------------------------------------------------------------- 2. load SOLPS (derive_data runs in __init__)
print(SOLPS_DIR)
solps = SOLPScase(SOLPS_DIR)

# Read Hermes-3 to get equilibrium data
# ---------------------------------------------------------------- 1. load the Hermes-3 case WITH guard cells
# Any converged run on the target grid works - it only supplies the mesh, geometry and the
# reference profiles for the comparison plots. Load.case_2D keeps x and y boundaries by default.
case = Load.case_2D(
    casepath=CASE_DIR,
    gridfilepath=GRID_IN,
    use_squash=True,     # writes/reuses BOUT.squash.nc - much faster than globbing BOUT.dmp.*
    unnormalise=True,    # SI units: m^-3, eV, m/s
    verbose=True,
)
case.extract_2d_tokamak_geometry()   # adds theta_idx + the metadata the selectors need

# interpolateSOLPStoHermes needs ONE time slice with dims (x, theta)
ds = case.ds.isel(t=-1)
m = ds.metadata

# Check
assert ds["Ne"].dims == ("x", "theta"), ds["Ne"].dims
print("\nshape (x, theta):", ds["Ne"].shape)
print({k: int(m[k]) for k in
       ["nx", "ny", "ny_inner", "MXG", "MYG", "ixseps1", "ixseps2",
        "jyseps1_1", "jyseps2_1", "jyseps1_2", "jyseps2_2"] if k in m})


/Users/zero/workspace/phd_work/hms_output/SOLPS_Luciani_off
- Reading case 260717_highrsl_6e20_coreOO
-----------------------
- Looking for squash file
- Squash file found. squash date 08/27/2026, 21:44:52, dmp file date 08/07/2026, 03:50:02


/Users/zero/workspace/phd_work/hms_output/xhermes/lib/python3.12/site-packages/boutdata/data.py:762: AlwaysWarning: While building x, y, z coordinate arrays, an exception occured: 'float' object cannot be interpreted as an integer
Evaluating non-scalar options not available
  alwayswarn(
/Users/zero/workspace/phd_work/hms_output/xBOUT/xbout/load.py:1294: UserWarning: Will drop all variables containing the dimensions 'closed_wall' because they are not recognised
  warn(



Unnormalising with xHermes

shape (x, theta): (40, 208)
{'nx': 40, 'ny': 200, 'ny_inner': 70, 'MXG': 2, 'MYG': 2, 'ixseps1': 20, 'ixseps2': 20, 'jyseps1_1': 19, 'jyseps2_1': 49, 'jyseps1_2': 109, 'jyseps2_2': 159}


In [9]:
# ------------------------------------------------- patch fpsi: b2fgmtry has it as all zeros
import scipy.interpolate
from code_comparison.interpolate import read_geqdsk

GEQDSK = os.path.join(ROOT, "46895_forward_ip.geqdsk")

if np.ptp(solps.g["fpsi"]) == 0:
    print("fpsi in SOLPS is zero so patching it ...")
    eq = read_geqdsk(open(GEQDSK).read())
    psi_spline = scipy.interpolate.RectBivariateSpline(eq["R1d"], eq["Z1d"], eq["psi"].T)
    solps.g["fpsi"] = psi_spline.ev(solps.g["R"], solps.g["Z"])
    print("patched fpsi from geqdsk")

f = solps.g["fpsi"]
assert f.shape == solps.g["R"].shape, (f.shape, solps.g["R"].shape)
assert np.isfinite(f).all()
print(f"fpsi range [{f.min():.5f}, {f.max():.5f}]")
for pol in (5, 50, 100, 150, 200):
    p = f[pol, 1:-1]
    assert np.all(np.diff(p) > 0) or np.all(np.diff(p) < 0), f"non-monotonic at pol {pol}"
print("fpsi monotonic radially at all sampled poloidal indices")

fpsi range [-0.03639, -0.00178]
fpsi monotonic radially at all sampled poloidal indices


In [ ]:
GEQDSK = "/Users/zero/workspace/phd_work/hms_output/46895_forward_ip.geqdsk"

# REGIONS = [
#     "outer_lower", "outer_upper", "inner_lower", "inner_upper",   # SOL legs
#     "core",
#     "inner_lower_pfr", "outer_lower_pfr", "inner_upper_pfr", "outer_upper_pfr",
# ]

# PARAMS = ["Ne", "Te", "Td+", "Vd+"]

REGIONS = [
    "outer_lower"
]

PARAMS = ["Ne"]
interp = interpolateSOLPStoHermes(ds, solps, equilibrium=GEQDSK)

for p in PARAMS:
    print(f"\n{'=' * 20} {p} {'=' * 20}")
    ds = interp.interpolate(
        p,
        regions=REGIONS,
        plot_lines=True,        # True -> overlays every Hermes/SOLPS field line on the grid (slow)
        plot_interp_debug=False, # True -> one 1D plot per ring (very slow, use on a single region)
    )

In [ ]:

interp.plot_1d_check(ds, "Ne")

# for p in PARAMS:
#     interp.plot_1d_check(ds, p)
#     plt.suptitle(p, y=1.01, fontsize=14)